In [ ]:
!pip install openai chromadb python-dotenv tiktoken langchain langchain-community rank_bm25 cohere numpy -q

In [9]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

True

In [10]:
import os
from pathlib import Path
import numpy as np
import chromadb
from openai import OpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter

client = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"


def ask(prompt, system_prompt=None, temperature=0, model="gpt-4o-mini", max_tokens=500):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    text = response.choices[0].message.content.strip()
    print(text)
    return text


def ask_raw(prompt, system_prompt=None, temperature=0, model="gpt-4o-mini", max_tokens=500):
    """Like ask() but returns just the text, no printing. For pipeline use."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, max_tokens=max_tokens
    )
    return response.choices[0].message.content.strip()


def get_embeddings(texts, model=EMBEDDING_MODEL):
    resp = client.embeddings.create(model=model, input=texts)
    return [d.embedding for d in resp.data]


def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

In [11]:
# Rebuild Part 1 pipeline (self-contained)
base_dir = Path.cwd()
candidates = [
    base_dir / "sample_docs",
    base_dir.parent / "4_rag_part1" / "sample_docs",
    Path("/Volumes/work/teach/genai-beginner/coaching_material/training_materials/4_rag_part1/sample_docs"),
]

sample_docs_dir = None
for c in candidates:
    if c.exists():
        sample_docs_dir = c
        break

if sample_docs_dir is None:
    raise FileNotFoundError("Could not find sample_docs/. Run notebook from 5_rag_part2 or update path.")

documents = []
for p in sorted(sample_docs_dir.glob("*.txt")):
    documents.append({"name": p.name, "content": p.read_text(encoding="utf-8")})

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

all_chunks = []
for doc in documents:
    chunks = splitter.split_text(doc["content"])
    for i, chunk in enumerate(chunks):
        all_chunks.append(
            {
                "id": f"{doc['name']}_chunk_{i}",
                "text": chunk,
                "source": doc["name"],
                "chunk_index": i,
            }
        )

chunk_embeddings = get_embeddings([c["text"] for c in all_chunks])

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("techstore_docs")
except Exception:
    pass

collection = chroma_client.create_collection("techstore_docs", metadata={"hnsw:space": "cosine"})
collection.add(
    ids=[c["id"] for c in all_chunks],
    embeddings=chunk_embeddings,
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"source": c["source"], "chunk_index": c["chunk_index"]} for c in all_chunks],
)


def search(query, n_results=3):
    q_emb = get_embeddings([query])[0]
    return collection.query(
        query_embeddings=[q_emb],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )


def rag_query(question, n_results=3):
    results = search(question, n_results=n_results)
    context = "\n\n---\n\n".join(results["documents"][0])
    answer = ask_raw(
        f"Context:\n---\n{context}\n---\n\nQuestion: {question}",
        system_prompt=(
            "You are a helpful assistant for TechStore. "
            "Answer ONLY from provided context. "
            "If answer is missing, say you do not have that information."
        ),
        temperature=0,
    )
    return answer, results

print(f"✅ Part 1 pipeline rebuilt: {len(all_chunks)} chunks in ChromaDB. Ready for Part 2.")

✅ Part 1 pipeline rebuilt: 31 chunks in ChromaDB. Ready for Part 2.


---
# Chapter 11: Hybrid Search — BM25 + Semantic
---
Semantic search finds meaning. Keyword search finds exact terms. Hybrid combines both.

In [32]:
# Semantic search sometimes struggles with exact product names
query = "ProMax 15"
results = search(query, n_results=3)

print(f"Semantic search for '{query}':")
for i in range(len(results['documents'][0])):
    score = 1 - results['distances'][0][i]
    print(f"  [{score:.3f}] \"{results['documents'][0][i][:100]}...\"")

print("\n⚠️  Semantic search captures 'meaning' but may rank exact name matches lower")
print("   because it embeds 'ProMax 15' as a general product concept.")

Semantic search for 'ProMax 15':
  [0.511] "TechStore Product Catalog — Q1 2025

Category: Laptops

ProMax 15 — $1,299
Our flagship laptop. 15.6..."
  [0.312] "WorkStation Pro 17 — $2,199
The powerhouse. 17-inch 4K display, Intel Xeon W processor, 64GB ECC RAM..."
  [0.289] "UltraWide 34 QHD — $599
34-inch curved ultrawide display, 3440x1440 resolution, 144Hz refresh rate. ..."

⚠️  Semantic search captures 'meaning' but may rank exact name matches lower
   because it embeds 'ProMax 15' as a general product concept.


In [42]:
from rank_bm25 import BM25Okapi

# Build BM25 index over the same chunks
chunk_texts = [c["text"] for c in all_chunks]
tokenized_chunks = [doc.lower().split() for doc in chunk_texts]
# print(tokenized_chunks[1])
bm25 = BM25Okapi(tokenized_chunks)

def search_bm25(query, n_results=3):
    """Keyword search using BM25."""
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = scores.argsort()[-n_results:][::-1]
    return [
        {"text": chunk_texts[i], "score": scores[i], "source": all_chunks[i]["source"]}
        for i in top_indices if scores[i] > 0
    ]

# Test BM25 with the same query
print("BM25 keyword search for 'ProMax 15':")
bm25_results = search_bm25("ProMax 15", n_results=3)
for r in bm25_results:
    print(f"  [{r['score']:.3f}] \"{r['text'][:100]}...\"")

print("\n✅ BM25 finds exact matches instantly — 'ProMax 15' appears in these chunks as text.")

BM25 keyword search for 'ProMax 15':
  [4.624] "TechStore Product Catalog — Q1 2025

Category: Laptops

ProMax 15 — $1,299
Our flagship laptop. 15.6..."
  [1.766] "CloudBook Air — $699
Ultra-portable for everyday use. 13.3-inch FHD display, Intel i5-1335U processo..."
  [1.392] "5. Holiday Return Policy
Purchases made between November 15 and December 31 of any calendar year are..."

✅ BM25 finds exact matches instantly — 'ProMax 15' appears in these chunks as text.


In [36]:
query = "ProMax 15"
query.lower().split()

['promax', '15']

In [37]:
tokenized_query = query.lower().split()
scores = bm25.get_scores(tokenized_query)

In [39]:
len(tokenized_chunks)

31

In [38]:
scores

array([1.38107617, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       4.62443998, 1.76625376, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 1.38107617, 0.        ,
       0.        , 1.3923218 , 0.        , 0.        , 0.        ,
       0.        ])

In [41]:
top_indices = scores.argsort()[-3:][::-1]
top_indices

array([15, 16, 26])

In [43]:
def search_hybrid(query, n_results=3, bm25_weight=0.4, semantic_weight=0.6):
    """Combine BM25 and semantic search results."""
    bm25_results = search_bm25(query, n_results=n_results * 2)
    semantic_results = search(query, n_results=n_results * 2)

    bm25_max = max(r['score'] for r in bm25_results) if bm25_results else 1
    combined = {}

    for r in bm25_results:
        key = r['text'][:80]
        normalized_score = r['score'] / bm25_max if bm25_max > 0 else 0
        combined[key] = {
            "text": r['text'],
            "source": r['source'],
            "bm25": normalized_score,
            "semantic": 0,
            "combined": bm25_weight * normalized_score,
        }

    for i in range(len(semantic_results['documents'][0])):
        text = semantic_results['documents'][0][i]
        key = text[:80]
        score = 1 - semantic_results['distances'][0][i]
        source = semantic_results['metadatas'][0][i]['source']

        if key in combined:
            combined[key]['semantic'] = score
            combined[key]['combined'] += semantic_weight * score
        else:
            combined[key] = {
                "text": text,
                "source": source,
                "bm25": 0,
                "semantic": score,
                "combined": semantic_weight * score,
            }

    ranked = sorted(combined.values(), key=lambda x: x['combined'], reverse=True)
    return ranked[:n_results]


query = "ProMax 15 return policy"
print(f"Query: '{query}'\n")

print("SEMANTIC ONLY:")
sem = search(query, n_results=3)
for i in range(len(sem['documents'][0])):
    print(f"  [{1-sem['distances'][0][i]:.3f}] \"{sem['documents'][0][i][:80]}...\"")

print("\nBM25 ONLY:")
for r in search_bm25(query, n_results=3):
    print(f"  [{r['score']:.2f}] \"{r['text'][:80]}...\"")

print("\nHYBRID (40% BM25 + 60% semantic):")
for r in search_hybrid(query, n_results=3):
    print(f"  [combined: {r['combined']:.3f}] (bm25: {r['bm25']:.2f}, sem: {r['semantic']:.2f})")
    print(f"    \"{r['text'][:80]}...\"")

print("\n💡 Hybrid finds chunks with exact 'ProMax 15' match AND relevant return policy content.")

Query: 'ProMax 15 return policy'

SEMANTIC ONLY:
  [0.546] "TechStore Return Policy
Last Updated: January 2025

1. Standard Return Policy
St..."
  [0.494] "9. Exceptions
The store manager has discretion to approve returns outside of sta..."
  [0.492] "5. Holiday Return Policy
Purchases made between November 15 and December 31 of a..."

BM25 ONLY:
  [6.40] "5. Holiday Return Policy
Purchases made between November 15 and December 31 of a..."
  [5.59] "2. Electronics Return Policy
All electronics (laptops, tablets, phones, monitors..."
  [4.62] "TechStore Product Catalog — Q1 2025

Category: Laptops

ProMax 15 — $1,299
Our f..."

HYBRID (40% BM25 + 60% semantic):
  [combined: 0.695] (bm25: 1.00, sem: 0.49)
    "5. Holiday Return Policy
Purchases made between November 15 and December 31 of a..."
  [combined: 0.644] (bm25: 0.87, sem: 0.49)
    "2. Electronics Return Policy
All electronics (laptops, tablets, phones, monitors..."
  [combined: 0.566] (bm25: 0.60, sem: 0.55)
    "TechStore Retu

---
# Chapter 12: Query Transformation
---
Users ask bad questions. Rewrite them before searching.

In [44]:
# Vague query → vague results
vague = "what's the deal with returns?"
print(f"VAGUE QUERY: '{vague}'")
results = search(vague, n_results=3)
for i in range(len(results['documents'][0])):
    score = 1 - results['distances'][0][i]
    print(f"  [{score:.3f}] \"{results['documents'][0][i][:80]}...\"")

VAGUE QUERY: 'what's the deal with returns?'
  [0.566] "9. Exceptions
The store manager has discretion to approve returns outside of sta..."
  [0.553] "TechStore Return Policy
Last Updated: January 2025

1. Standard Return Policy
St..."
  [0.494] "5. Holiday Return Policy
Purchases made between November 15 and December 31 of a..."


In [45]:
def rewrite_query(query):
    """Use LLM to rewrite a vague query into a specific, search-friendly one."""
    return ask_raw(
        f"Rewrite this search query to be more specific and detailed. "
        f"Output ONLY the rewritten query, nothing else.\n\n"
        f"Original: {query}",
        temperature=0,
    )

vague = "what's the deal with returns?"
rewritten = rewrite_query(vague)

print(f"Original:  '{vague}'")
print(f"Rewritten: '{rewritten}'\n")

print("RESULTS WITH REWRITTEN QUERY:")
results = search(rewritten, n_results=3)
for i in range(len(results['documents'][0])):
    score = 1 - results['distances'][0][i]
    print(f"  [{score:.3f}] \"{results['documents'][0][i][:80]}...\"")

print("\n💡 Rewriting cost: 1 extra LLM call (~$0.0001). Quality improvement: significant.")

Original:  'what's the deal with returns?'
Rewritten: 'What are the specific policies and procedures for returning items, including timeframes, conditions for eligibility, and any associated fees?'

RESULTS WITH REWRITTEN QUERY:
  [0.504] "TechStore Return Policy
Last Updated: January 2025

1. Standard Return Policy
St..."
  [0.479] "5. Holiday Return Policy
Purchases made between November 15 and December 31 of a..."
  [0.475] "9. Exceptions
The store manager has discretion to approve returns outside of sta..."

💡 Rewriting cost: 1 extra LLM call (~$0.0001). Quality improvement: significant.


In [46]:
def step_back_query(query):
    """Generate a broader 'step-back' question for better context retrieval."""
    return ask_raw(
        f"Given this specific question, generate a broader question that would "
        f"retrieve useful background context.\n\n"
        f"Specific: {query}\n"
        f"Broader question:",
        temperature=0,
    )

specific = "Can I return a ProMax 15 after 20 days if I have the receipt?"
broader = step_back_query(specific)

print(f"Specific: '{specific}'")
print(f"Step-back: '{broader}'\n")

print("SPECIFIC QUERY results:")
r1 = search(specific, n_results=2)
for i in range(len(r1['documents'][0])):
    print(f"  [{1-r1['distances'][0][i]:.3f}] \"{r1['documents'][0][i][:80]}...\"")

print("\nSTEP-BACK QUERY results:")
r2 = search(broader, n_results=2)
for i in range(len(r2['documents'][0])):
    print(f"  [{1-r2['distances'][0][i]:.3f}] \"{r2['documents'][0][i][:80]}...\"")

print("\n💡 Step-back finds the general policy. Specific finds the electronics detail.")
print("   Together they give the LLM everything needed to answer accurately.")

Specific: 'Can I return a ProMax 15 after 20 days if I have the receipt?'
Step-back: 'What is the return policy for electronics at major retailers, including time frames and conditions for returns?'

SPECIFIC QUERY results:
  [0.538] "TechStore Return Policy
Last Updated: January 2025

1. Standard Return Policy
St..."
  [0.516] "2. Electronics Return Policy
All electronics (laptops, tablets, phones, monitors..."

STEP-BACK QUERY results:
  [0.679] "2. Electronics Return Policy
All electronics (laptops, tablets, phones, monitors..."
  [0.601] "TechStore Return Policy
Last Updated: January 2025

1. Standard Return Policy
St..."

💡 Step-back finds the general policy. Specific finds the electronics detail.
   Together they give the LLM everything needed to answer accurately.


---
# Chapter 13: HyDE — Hypothetical Document Embeddings
---
Generate a fake answer, embed it, search with that instead of the question.

In [48]:
def search_hyde(query, n_results=3):
    """HyDE: generate hypothetical answer, use it for search."""
    hypothetical = ask_raw(
        f"Answer this question in 2-3 sentences as if you were a company policy document:\n{query}",
        temperature=0.3,
    )

    hyde_embedding = get_embeddings([hypothetical])[0]
    results = collection.query(
        query_embeddings=[hyde_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    return results, hypothetical


query = "What happens if my package is broken?"

print(f"Query: '{query}'\n")

# print("NORMAL SEARCH (embed the question):")
# normal = search(query, n_results=3)
# for i in range(len(normal['documents'][0])):
#     score = 1 - normal['distances'][0][i]
#     print(f"  [{score:.3f}] \"{normal['documents'][0][i][:80]}...\"")

print("\nHyDE SEARCH (embed a hypothetical answer):")
hyde_results, hypothetical = search_hyde(query, n_results=3)
print(f"  Hypothetical answer: \"{hypothetical}...\"")
print()
for i in range(len(hyde_results['documents'][0])):
    score = 1 - hyde_results['distances'][0][i]
    print(f"  [{score:.3f}] \"{hyde_results['documents'][0][i][:80]}...\"")

print("\n💡 The hypothetical answer sounds like a 'document' — its embedding")
print("   matches stored documents better than a short question does.")

Query: 'What happens if my package is broken?'


HyDE SEARCH (embed a hypothetical answer):
  Hypothetical answer: "If your package arrives damaged, please contact our customer service team within 48 hours of delivery. We will require photographic evidence of the damage to process your claim and arrange for a replacement or refund, as per our return policy. Please retain all packaging materials until the issue is resolved...."

  [0.756] "6. Damaged Items
Items that arrive damaged or are found to be defective must be ..."
  [0.476] "2. Electronics Return Policy
All electronics (laptops, tablets, phones, monitors..."
  [0.414] "TechStore Return Policy
Last Updated: January 2025

1. Standard Return Policy
St..."

💡 The hypothetical answer sounds like a 'document' — its embedding
   matches stored documents better than a short question does.


---
# Chapter 14: Re-ranking
---
Retrieve many with fast search, then re-score with a smarter model.

Note: We'll implement re-ranking using the LLM as a re-ranker (no Cohere API needed).
For production, use Cohere's rerank API or sentence-transformers cross-encoders.

In [51]:
def rerank_with_llm(query, chunks, top_n=3):
    """Re-rank chunks using the LLM as a relevance scorer.

    Production alternative: Cohere rerank API or cross-encoder models.
    LLM re-ranking is slower but doesn't need extra API keys for learning.
    """
    scored = []

    for chunk in chunks:
        score_text = ask_raw(
            f"On a scale of 0 to 10, how relevant is this document chunk "
            f"for answering the question?\n\n"
            f"Question: {query}\n\n"
            f"Chunk: {chunk['text'][:300]}\n\n"
            f"Reply with ONLY a number from 0 to 10.",
            temperature=0,
        )
        try:
            score = float(score_text.strip())
        except Exception:
            score = 0
        scored.append({**chunk, "rerank_score": score})

    scored.sort(key=lambda x: x['rerank_score'], reverse=True)
    return scored[:top_n]


query = "What benefits does TechStore provide to employees?"

print(f"Query: '{query}'\n")
print("Step 1: Retrieve top 8 with embedding search:")
initial = search(query, n_results=8)
initial_chunks = []
for i in range(len(initial['documents'][0])):
    score = 1 - initial['distances'][0][i]
    chunk = {"text": initial['documents'][0][i], "source": initial['metadatas'][0][i]['source'], "embed_score": score}
    initial_chunks.append(chunk)
    print(f"  [{i+1}] embed: {score:.3f} | {chunk['source']}: \"{chunk['text']}...\"")

print("\nStep 2: Re-rank top 8 → keep top 3:")
reranked = rerank_with_llm(query, initial_chunks, top_n=3)
for i, chunk in enumerate(reranked):
    print(f"  [{i+1}] rerank: {chunk['rerank_score']}/10 (was embed: {chunk['embed_score']:.3f})")
    print(f"       {chunk['source']}: \"{chunk['text']}...\"")

print("\n💡 Re-ranking changes the order — chunks that truly answer the question rise to the top.")
print("   In production, use Cohere rerank API (faster, cheaper than LLM re-ranking).")

Query: 'What benefits does TechStore provide to employees?'

Step 1: Retrieve top 8 with embedding search:
  [1] embed: 0.679 | employee_handbook.txt: "Health insurance coverage begins on the first day of the month following 30 days of employment. TechStore offers three health plan options: Basic (employee pays 10% of premium), Standard (employee pays 20%), and Premium (employee pays 30%). Dental and vision insurance are included in all plans. The company matches 401(k) contributions up to 4% of salary. Employees receive a 25% discount on all TechStore products, up to $2,000 per year in total discount value..."
  [2] embed: 0.496 | employee_handbook.txt: "TechStore Employee Handbook — HR Policies
Version 3.2 | Effective March 2025

Chapter 1: Paid Time Off (PTO)

Full-time employees accrue PTO at the following rates based on years of service:
- 0-2 years: 15 days per year (1.25 days per month)
- 3-5 years: 20 days per year (1.67 days per month)
- 6+ years: 25 days per year (2.08 days p

---
# Chapter 15: Multi-Query RAG
---
Break complex questions into sub-questions, search for each, combine results.

In [52]:
def multi_query_search(query, n_results_per_query=3):
    """Break a complex query into sub-queries, search each, merge results."""
    sub_queries_text = ask_raw(
        f"Break this complex question into 2-3 simpler sub-questions "
        f"that together would fully answer the original.\n\n"
        f"Question: {query}\n\n"
        f"Output each sub-question on a new line. No numbering, no bullets.",
        temperature=0,
    )
    sub_queries = [q.strip() for q in sub_queries_text.strip().split("\n") if q.strip()]

    all_results = {}
    for sq in sub_queries:
        results = search(sq, n_results=n_results_per_query)
        for i in range(len(results['documents'][0])):
            text = results['documents'][0][i]
            key = text[:80]
            score = 1 - results['distances'][0][i]
            if key not in all_results or all_results[key]['score'] < score:
                all_results[key] = {
                    "text": text,
                    "source": results['metadatas'][0][i]['source'],
                    "score": score,
                    "matched_query": sq,
                }

    merged = sorted(all_results.values(), key=lambda x: x['score'], reverse=True)
    return merged, sub_queries


query = "Compare the return policies for electronics versus software products"

print(f"Complex query: '{query}'\n")

merged, sub_queries = multi_query_search(query)

print(f"Generated {len(sub_queries)} sub-queries:")
for i, sq in enumerate(sub_queries):
    print(f"  {i+1}. {sq}")

print(f"\nMerged results ({len(merged)} unique chunks):")
for r in merged[:5]:
    print(f"  [{r['score']:.3f}] {r['source']} (matched: \"{r['matched_query'][:50]}\")")
    print(f"    \"{r['text'][:80]}...\"")

print("\n💡 Single search for 'compare electronics vs software' might miss one side.")
print("   Multi-query ensures both topics are retrieved.")

Complex query: 'Compare the return policies for electronics versus software products'

Generated 3 sub-queries:
  1. What are the typical return policies for electronics products?
  2. What are the typical return policies for software products?
  3. How do the return policies for electronics and software products differ in terms of time limits and conditions?

Merged results (4 unique chunks):
  [0.730] return_policy.txt (matched: "What are the typical return policies for electroni")
    "2. Electronics Return Policy
All electronics (laptops, tablets, phones, monitors..."
  [0.657] return_policy.txt (matched: "What are the typical return policies for electroni")
    "TechStore Return Policy
Last Updated: January 2025

1. Standard Return Policy
St..."
  [0.637] return_policy.txt (matched: "What are the typical return policies for software ")
    "3. Software and Digital Products
Software products are non-refundable once the a..."
  [0.571] return_policy.txt (matched: "What are the typic

---
# Chapter 16: Metadata Filtering — Deep Dive
---
In Part 1 we did basic filtering. Here we add richer metadata and show production patterns.

In [53]:
# Rebuild collection with RICHER metadata
try:
    chroma_client.delete_collection("techstore_rich")
except Exception:
    pass

rich_collection = chroma_client.create_collection("techstore_rich", metadata={"hnsw:space": "cosine"})

category_map = {
    "return_policy.txt": {"category": "policy", "department": "customer_service", "access": "public"},
    "employee_handbook.txt": {"category": "hr", "department": "human_resources", "access": "internal"},
    "product_catalog.txt": {"category": "product", "department": "sales", "access": "public"},
}

enriched_metadatas = []
for c in all_chunks:
    meta = {
        "source": c["source"],
        "chunk_index": c["chunk_index"],
        "last_updated": "2025-01-15" if "return" in c["source"] else "2025-03-01",
        **category_map.get(c["source"], {}),
    }
    enriched_metadatas.append(meta)

rich_collection.add(
    ids=[c["id"] for c in all_chunks],
    embeddings=chunk_embeddings,
    documents=[c["text"] for c in all_chunks],
    metadatas=enriched_metadatas,
)

print(f"✅ Rich collection: {rich_collection.count()} chunks with enriched metadata\n")

query = "What policies apply to me?"
q_emb = get_embeddings([query])[0]

public = rich_collection.query(query_embeddings=[q_emb], n_results=3, where={"access": "public"}, include=["metadatas"])
print("PUBLIC docs only:", [f"{m['source']} ({m['category']})" for m in public['metadatas'][0]])

internal = rich_collection.query(query_embeddings=[q_emb], n_results=3, where={"access": "internal"}, include=["metadatas"])
print("INTERNAL docs only:", [f"{m['source']} ({m['category']})" for m in internal['metadatas'][0]])

hr = rich_collection.query(query_embeddings=[q_emb], n_results=3, where={"department": "human_resources"}, include=["metadatas"])
print("HR dept only:", [f"{m['source']} ({m['category']})" for m in hr['metadatas'][0]])

combo = rich_collection.query(
    query_embeddings=[q_emb], n_results=3,
    where={"$and": [{"access": "public"}, {"category": "policy"}]},
    include=["metadatas"],
)
print("Public + Policy:", [f"{m['source']} ({m['category']})" for m in combo['metadatas'][0]])

print("\n💡 Same query, different results based on who's asking.")
print("   This is how enterprise RAG implements access control.")

✅ Rich collection: 31 chunks with enriched metadata

PUBLIC docs only: ['return_policy.txt (policy)', 'return_policy.txt (policy)', 'return_policy.txt (policy)']
INTERNAL docs only: ['employee_handbook.txt (hr)', 'employee_handbook.txt (hr)', 'employee_handbook.txt (hr)']
HR dept only: ['employee_handbook.txt (hr)', 'employee_handbook.txt (hr)', 'employee_handbook.txt (hr)']
Public + Policy: ['return_policy.txt (policy)', 'return_policy.txt (policy)', 'return_policy.txt (policy)']

💡 Same query, different results based on who's asking.
   This is how enterprise RAG implements access control.


---
# Chapter 17: Parent-Child Retrieval
---
Search with small chunks (precise). Send big chunks to the LLM (context).

In [12]:
len(documents)

3

In [15]:
documents

[{'name': 'employee_handbook.txt',
  'content': "TechStore Employee Handbook — HR Policies\nVersion 3.2 | Effective March 2025\n\nChapter 1: Paid Time Off (PTO)\n\nFull-time employees accrue PTO at the following rates based on years of service:\n- 0-2 years: 15 days per year (1.25 days per month)\n- 3-5 years: 20 days per year (1.67 days per month)\n- 6+ years: 25 days per year (2.08 days per month)\n\nPTO requests must be submitted at least 2 weeks in advance through the HR portal. Requests during peak retail periods (Black Friday week, last two weeks of December) require manager approval at least 30 days in advance. Unused PTO may be carried over to the following year, up to a maximum of 5 days. PTO balances exceeding the carry-over limit are forfeited on January 1. Upon separation from the company, accrued but unused PTO is paid out at the employee's current rate.\n\nChapter 2: Remote Work Policy\n\nCorporate and technology employees may work remotely up to 3 days per week with mana

In [16]:
# Build parent-child index
from langchain.text_splitter import RecursiveCharacterTextSplitter

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0, separators=["\n\n", "\n", ". ", " "])
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30, separators=["\n\n", "\n", ". ", " "])

parents = []
children = []

for doc in documents:
    parent_chunks = parent_splitter.split_text(doc["content"])
    for p_idx, parent_text in enumerate(parent_chunks):
        parent_id = f"{doc['name']}_parent_{p_idx}"
        parents.append({"id": parent_id, "text": parent_text, "source": doc["name"]})

        child_chunks = child_splitter.split_text(parent_text)
        for c_idx, child_text in enumerate(child_chunks):
            children.append({
                "id": f"{parent_id}_child_{c_idx}",
                "text": child_text,
                "parent_id": parent_id,
                "source": doc["name"],
            })

print(f"Parents: {len(parents)} (large chunks, ~1000 chars)")
print(f"Children: {len(children)} (small chunks, ~200 chars)")

child_texts = [c["text"] for c in children]
child_embeddings = get_embeddings(child_texts)

try:
    chroma_client.delete_collection("children")
except Exception:
    pass

child_collection = chroma_client.create_collection("children", metadata={"hnsw:space": "cosine"})
child_collection.add(
    ids=[c["id"] for c in children],
    embeddings=child_embeddings,
    documents=[c["text"] for c in children],
    metadatas=[{"parent_id": c["parent_id"], "source": c["source"]} for c in children],
)

parent_lookup = {p["id"]: p["text"] for p in parents}

def search_parent_child(query, n_results=3):
    q_emb = get_embeddings([query])[0]
    results = child_collection.query(query_embeddings=[q_emb], n_results=n_results, include=["documents", "metadatas", "distances"])

    seen_parents = set()
    parent_results = []
    for i in range(len(results['documents'][0])):
        pid = results['metadatas'][0][i]['parent_id']
        if pid not in seen_parents:
            seen_parents.add(pid)
            parent_results.append({
                "child_text": results['documents'][0][i],
                "parent_text": parent_lookup[pid],
                "score": 1 - results['distances'][0][i],
                "source": results['metadatas'][0][i]['source'],
            })
    return parent_results


query = "What is the electronics return window?"
results = search_parent_child(query, n_results=3)

print(f"\nQuery: '{query}'\n")
for i, r in enumerate(results[:2]):
    print(f"MATCH {i+1} (score: {r['score']:.3f}, source: {r['source']})")
    print(f"  Child matched ({len(r['child_text'])} chars): \"{r['child_text'][:100]}...\"")
    print(f"  Parent sent to LLM ({len(r['parent_text'])} chars): \"{r['parent_text'][:150]}...\"")
    print()

print("💡 Child chunk is precise (matches the question).")
print("   Parent chunk has full context (conditions, exceptions, process).")
print("   LLM gets the whole picture, not just a fragment.")

Parents: 14 (large chunks, ~1000 chars)
Children: 97 (small chunks, ~200 chars)

Query: 'What is the electronics return window?'

MATCH 1 (score: 0.726, source: return_policy.txt)
  Child matched (28 chars): "2. Electronics Return Policy..."
  Parent sent to LLM (862 chars): "TechStore Return Policy
Last Updated: January 2025

1. Standard Return Policy
Standard items may be returned within 30 days of the original purchase d..."

💡 Child chunk is precise (matches the question).
   Parent chunk has full context (conditions, exceptions, process).
   LLM gets the whole picture, not just a fragment.


---
# Chapter 18: Conversational RAG
---
Handle follow-up questions by rewriting them with chat history context.

In [17]:
def conversational_rag(question, chat_history=None):
    """RAG with follow-up question handling."""
    search_query = question
    if chat_history and len(chat_history) > 0:
        history_text = "\n".join(f"{m['role']}: {m['content']}" for m in chat_history[-4:])
        search_query = ask_raw(
            f"Given this conversation history:\n{history_text}\n\n"
            f"Rewrite this follow-up question to be a standalone question "
            f"that makes sense WITHOUT the history:\n'{question}'\n\n"
            f"Output ONLY the rewritten question.",
            temperature=0,
        )
        print(f"  Rewritten: '{search_query}'")

    results = search(search_query, n_results=3)

    context_parts = []
    for i in range(len(results['documents'][0])):
        context_parts.append(results['documents'][0][i])
    context = "\n\n---\n\n".join(context_parts)

    history_prompt = ""
    if chat_history:
        history_prompt = "Previous conversation:\n" + "\n".join(
            f"{m['role']}: {m['content']}" for m in chat_history[-4:]
        ) + "\n\n"

    answer = ask_raw(
        f"{history_prompt}"
        f"Context:\n---\n{context}\n---\n\n"
        f"Question: {question}",
        system_prompt="Answer using ONLY the provided context. If not in context, say so.",
    )

    return answer, search_query


chat_history = []

print("═" * 60)
print("Turn 1:")
q1 = "What is the return policy?"
print(f"User: {q1}")
a1, _ = conversational_rag(q1, chat_history)
print(f"AI: {a1}")
chat_history.extend([{"role": "user", "content": q1}, {"role": "assistant", "content": a1}])

print(f"\n{'═' * 60}")
print("Turn 2 (follow-up):")
q2 = "What about electronics?"
print(f"User: {q2}")
a2, rewritten2 = conversational_rag(q2, chat_history)
print(f"AI: {a2}")
chat_history.extend([{"role": "user", "content": q2}, {"role": "assistant", "content": a2}])

print(f"\n{'═' * 60}")
print("Turn 3 (another follow-up):")
q3 = "And if it arrives damaged?"
print(f"User: {q3}")
a3, rewritten3 = conversational_rag(q3, chat_history)
print(f"AI: {a3}")

print("\n💡 Each follow-up was rewritten using chat history BEFORE searching.")
print(f"   'What about electronics?' → '{rewritten2}' (or similar)")
print("   Without rewriting, we'd search for 'What about electronics?' and get product specs!")

════════════════════════════════════════════════════════════
Turn 1:
User: What is the return policy?
AI: The return policy allows standard items to be returned within 30 days of the original purchase date, requiring a valid receipt or proof of purchase. Items must be in unused, resalable condition with all original tags and packaging intact. Opened items may incur a 15% restocking fee at the manager's discretion. 

During the holiday season, purchases made between November 15 and December 31 can be returned until January 31 of the following year, but all other terms still apply. Electronics have a standard 15-day return policy year-round. 

Exceptions can be made at the store manager's discretion, limited to one per customer per calendar year. Items from third-party sellers follow their individual return policies, and clearance items marked as "Final Sale" are non-returnable and non-refundable.

════════════════════════════════════════════════════════════
Turn 2 (follow-up):
User: Wha

---
# Chapter 19: Multi-Document RAG
---
Route queries to the right collection, or search across multiple collections.

In [18]:
collections = {}
for doc in documents:
    name = doc['name'].replace('.txt', '')
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass

    col = chroma_client.create_collection(name, metadata={"hnsw:space": "cosine"})
    chunks = splitter.split_text(doc["content"])
    embeddings = get_embeddings(chunks)
    col.add(
        ids=[f"{name}_{i}" for i in range(len(chunks))],
        embeddings=embeddings,
        documents=chunks,
    )
    collections[name] = col
    print(f"✅ Collection '{name}': {col.count()} chunks")


def route_and_search(query, n_results=3):
    """Use LLM to pick the right collection, then search it."""
    col_names = list(collections.keys())

    chosen = ask_raw(
        f"Which collection should I search to answer: '{query}'?\n"
        f"Options: {', '.join(col_names)}\n"
        f"Reply with ONLY the collection name.",
        temperature=0,
    ).strip()

    col = collections.get(chosen)
    if not col:
        for name in col_names:
            if name in chosen.lower() or chosen.lower() in name:
                col = collections[name]
                chosen = name
                break

    if not col:
        chosen = col_names[0]
        col = collections[chosen]

    q_emb = get_embeddings([query])[0]
    results = col.query(query_embeddings=[q_emb], n_results=n_results, include=["documents", "distances"])
    return results, chosen


for q in ["How many PTO days do I get?", "Can I return a laptop?", "What laptop has 64GB RAM?"]:
    results, chosen = route_and_search(q)
    score = 1 - results['distances'][0][0]
    print(f"Q: '{q}'")
    print(f"  → Routed to: {chosen} | Top score: {score:.3f}")
    print(f"  → \"{results['documents'][0][0][:80]}...\"\n")

✅ Collection 'employee_handbook': 15 chunks
✅ Collection 'product_catalog': 7 chunks
✅ Collection 'return_policy': 9 chunks
Q: 'How many PTO days do I get?'
  → Routed to: employee_handbook | Top score: 0.621
  → "PTO requests must be submitted at least 2 weeks in advance through the HR portal..."

Q: 'Can I return a laptop?'
  → Routed to: return_policy | Top score: 0.586
  → "2. Electronics Return Policy
All electronics (laptops, tablets, phones, monitors..."

Q: 'What laptop has 64GB RAM?'
  → Routed to: product_catalog | Top score: 0.565
  → "TechStore Product Catalog — Q1 2025

Category: Laptops

ProMax 15 — $1,299
Our f..."



---
# Chapter 20: RAG Evaluation Framework
---
Measuring retrieval quality and answer quality separately.

In [19]:
eval_dataset = [
    {"question": "What is the return window for electronics?", "ground_truth": "15 days", "expected_source": "return_policy.txt"},
    {"question": "How many PTO days after 4 years of service?", "ground_truth": "20 days per year", "expected_source": "employee_handbook.txt"},
    {"question": "What laptop is best for machine learning?", "ground_truth": "WorkStation Pro 17", "expected_source": "product_catalog.txt"},
    {"question": "Are gift cards refundable?", "ground_truth": "No, gift cards are non-refundable", "expected_source": "return_policy.txt"},
    {"question": "What is the remote work policy for retail employees?", "ground_truth": "Retail store employees are not eligible for remote work", "expected_source": "employee_handbook.txt"},
    {"question": "What monitor supports USB-C charging?", "ground_truth": "ClearView 27 4K with 65W power delivery", "expected_source": "product_catalog.txt"},
]

print(f"Evaluating {len(eval_dataset)} questions:\n")
print(f"{'Question':<45} {'Retrieval':>10} {'Faithful':>10} {'Correct':>10}")
print("─" * 80)

for item in eval_dataset:
    q = item["question"]

    results = search(q, n_results=3)
    top_source = results['metadatas'][0][0]['source']

    retrieval_correct = top_source == item["expected_source"]

    context = "\n\n".join(results['documents'][0])
    answer = ask_raw(
        f"Context:\n{context}\n\nQuestion: {q}",
        system_prompt="Answer ONLY from context. If not found, say 'Not found'.",
    )

    faithful = "not found" not in answer.lower() or "not found" in item["ground_truth"].lower()
    correct = item["ground_truth"].lower() in answer.lower() or any(
        word in answer.lower() for word in item["ground_truth"].lower().split()
        if len(word) > 3
    )

    r_icon = "✅" if retrieval_correct else "❌"
    f_icon = "✅" if faithful else "❌"
    c_icon = "✅" if correct else "⚠️"

    print(f"{q[:43]:<45} {r_icon:>10} {f_icon:>10} {c_icon:>10}")

print("\n💡 This is a simplified evaluation. Next use Chapter 21 debugging workflow to improve weak spots.")

Evaluating 6 questions:

Question                                       Retrieval   Faithful    Correct
────────────────────────────────────────────────────────────────────────────────
What is the return window for electronics?             ✅          ✅          ✅
How many PTO days after 4 years of service?            ✅          ✅          ✅
What laptop is best for machine learning?              ✅          ✅          ✅
Are gift cards refundable?                             ✅          ✅          ✅
What is the remote work policy for retail e            ✅          ❌         ⚠️
What monitor supports USB-C charging?                  ✅          ✅          ✅

💡 This is a simplified evaluation. Next use Chapter 21 debugging workflow to improve weak spots.


---
# Chapter 21: Debugging RAG
---
Is it a retrieval problem or a generation problem? Use metrics to decide.

In [20]:
def diagnose_rag(question, expected_answer=None, n_chunks=5):
    """Full diagnostic: check retrieval, then generation."""
    print(f"🔍 Diagnosing: '{question}'")
    print("═" * 60)

    results = search(question, n_results=n_chunks)
    print("\n📋 RETRIEVAL CHECK:")
    scores = []
    for i in range(len(results['documents'][0])):
        score = 1 - results['distances'][0][i]
        scores.append(score)
        src = results['metadatas'][0][i]['source']
        tag = "🟢" if score > 0.5 else "🟡" if score > 0.35 else "🔴"
        print(f"  {tag} [{score:.3f}] {src}: \"{results['documents'][0][i][:60]}...\"")

    avg_score = sum(scores[:3]) / min(3, len(scores))
    print(f"\n  Avg top-3 score: {avg_score:.3f}")

    if avg_score < 0.35:
        print("  ❌ RETRIEVAL PROBLEM — relevant chunks not found")
        print("     Try: query rewriting, hybrid search, check if info is in docs")
        return

    context = "\n\n".join(results['documents'][0][:3])
    answer = ask_raw(
        f"Context:\n{context}\n\nQuestion: {question}",
        system_prompt="Answer ONLY from context. If not found, say 'Not found'.",
    )

    print("\n🤖 GENERATION CHECK:")
    print(f"  Answer: \"{answer[:150]}\"")

    uses_context = any(word in answer.lower() for word in context.lower().split()[:20] if len(word) > 4)
    says_not_found = "not found" in answer.lower() or "don't have" in answer.lower()

    if expected_answer:
        correct = expected_answer.lower() in answer.lower()
        print(f"  Expected: \"{expected_answer}\"")
        print(f"  Correct: {'✅' if correct else '❌'}")

    if says_not_found and avg_score > 0.5:
        print("  ⚠️  GENERATION PROBLEM — relevant chunks found but LLM says 'not found'")
        print("     Try: strengthen system prompt, reduce temperature, use better model")
    elif not uses_context and not says_not_found:
        print("  ⚠️  HALLUCINATION RISK — answer may not be grounded in context")
        print("     Try: add 'Answer ONLY from context' instruction")
    else:
        print("  ✅ Answer appears grounded in retrieved context")


diagnose_rag("What is the return window for electronics?", expected_answer="15 days")
print()
diagnose_rag("What is TechStore's stock price?")
print()
diagnose_rag("Can I send back my gadget?", expected_answer="return")

🔍 Diagnosing: 'What is the return window for electronics?'
════════════════════════════════════════════════════════════

📋 RETRIEVAL CHECK:
  🟢 [0.704] return_policy.txt: "2. Electronics Return Policy
All electronics (laptops, table..."
  🟢 [0.561] return_policy.txt: "5. Holiday Return Policy
Purchases made between November 15 ..."
  🟢 [0.546] return_policy.txt: "TechStore Return Policy
Last Updated: January 2025

1. Stand..."
  🟢 [0.501] return_policy.txt: "9. Exceptions
The store manager has discretion to approve re..."
  🟡 [0.421] return_policy.txt: "7. Refund Processing
Refunds are processed to the original p..."

  Avg top-3 score: 0.604

🤖 GENERATION CHECK:
  Answer: "The return window for electronics is 15 days from the date of purchase."
  Expected: "15 days"
  Correct: ✅
  ✅ Answer appears grounded in retrieved context

🔍 Diagnosing: 'What is TechStore's stock price?'
════════════════════════════════════════════════════════════

📋 RETRIEVAL CHECK:
  🟢 [0.521] employee_handbook

---
# Chapters 23-25: Production Infrastructure
---
These chapters are **conceptual** — covered on the web page with comparison tables and checklists. No code demos needed here because:
- **Ch 23 (Vector DB Comparison):** You'd need accounts on Pinecone, Weaviate, etc. The web page has the comparison table.
- **Ch 24 (Scaling RAG):** Requires large datasets to demonstrate. Concepts are on the web page.
- **Ch 25 (RAG in Production):** Deployment patterns are covered in the LLMOps section later.

Key takeaways:
- **ChromaDB** → prototyping. **Pinecone** → managed production. **pgvector** → already using PostgreSQL.
- **Scaling:** batch embed, incremental updates, shard large collections
- **Production:** cache frequent queries, monitor retrieval scores, version your knowledge base, run evaluation checks weekly

---
# Summary
---

| Chapter | Technique | When to use |
|---|---|---|
| 11. Hybrid Search | BM25 + semantic | Users search for exact names/codes + meaning |
| 12. Query Transformation | Rewrite vague queries | Always — highest ROI improvement |
| 13. HyDE | Search with fake answer | Short vague queries against long documents |
| 14. Re-ranking | Retrieve many, re-score | When top-K results aren't precise enough |
| 15. Multi-Query | Break into sub-questions | Complex questions spanning multiple topics |
| 16. Metadata Filtering | Filter before search | Access control, date ranges, categories |
| 17. Parent-Child | Search small, send big | Need both precision and context |
| 18. Conversational | Rewrite follow-ups | Chatbot / multi-turn applications |
| 19. Multi-Document | Route to right collection | Multiple document types / knowledge bases |
| 20-21. Evaluation | Measure + debug | Before every production deployment |

### Next → Frameworks (LangChain & LlamaIndex)